In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from rank_bm25 import BM25Okapi

In [2]:
save_dir = "models/bm25"
os.makedirs(save_dir, exist_ok=True)

In [3]:
print("Đang tải dữ liệu bài báo (Có stopwords)...")
df_with_sw = pd.read_pickle("data/news_dataset_df_prep2.pkl")
print(f"-> Tải thành công {len(df_with_sw)} bài báo có stopwords.")

print("Đang tải dữ liệu bài báo (Không có stopwords)...")
df_no_sw = pd.read_pickle("data/news_dataset_df_prep3.pkl")
print(f"-> Tải thành công {len(df_no_sw)} bài báo không có stopwords.")

Đang tải dữ liệu bài báo (Có stopwords)...
-> Tải thành công 160254 bài báo có stopwords.
Đang tải dữ liệu bài báo (Không có stopwords)...
-> Tải thành công 160254 bài báo không có stopwords.


In [4]:
text_column = "combined_processed"

print("Đang tokenize corpus (Có stopwords)...")
tokenized_corpus_with_sw = [str(doc).split() for doc in df_with_sw[text_column].fillna("")]

print("Đang tokenize corpus (Không có stopwords)...")
tokenized_corpus_no_sw = [str(doc).split() for doc in df_no_sw[text_column].fillna("")]

print("Đang xây dựng BM25 model (Có stopwords)...")
bm25_model_with_sw = BM25Okapi(tokenized_corpus_with_sw)

print("Đang xây dựng BM25 model (Không có stopwords)...")
bm25_model_no_sw = BM25Okapi(tokenized_corpus_no_sw)

print("\n-> Đã xây dựng thành công cả 2 mô hình BM25!")

Đang tokenize corpus (Có stopwords)...
Đang tokenize corpus (Không có stopwords)...
Đang xây dựng BM25 model (Có stopwords)...
Đang xây dựng BM25 model (Không có stopwords)...

-> Đã xây dựng thành công cả 2 mô hình BM25!


In [5]:
def search_bm25(query_text, k, bm25_model, df_documents, model_name=""):
    print(f"\n{'='*80}")
    print(f"**Truy vấn [{model_name}]:** '{query_text}'")
    
    tokenized_query = query_text.split()
    scores = bm25_model.get_scores(tokenized_query)
    
    top_k_idx = np.argsort(scores)[::-1][:k]
    
    results = df_documents.iloc[top_k_idx].copy()
    results["bm25_score"] = scores[top_k_idx]
    
    return results

def display_bm25_results(results):
    for rank, (_, row) in enumerate(results.iterrows()):
        print("-" * 80)
        print(f"Rank {rank+1} | BM25 Score: {row['bm25_score']:.4f}")
        print(f"Tiêu đề: {row.get('title', 'Không có title')}")
        print(f"Nguồn: {row.get('source', '')} | Chủ đề: {row.get('topic', '')}")
        print(f"Nội dung: {str(row.get('content', ''))[:200]}...")

In [6]:
query = "kinh tế việt nam"
k_results = 5

# --- TEST MÔ HÌNH CÓ STOPWORDS ---
results_with_sw = search_bm25(
    query_text=query,
    k=k_results,
    bm25_model=bm25_model_with_sw,
    df_documents=df_with_sw,
    model_name="CÓ Stopwords"
)
display_bm25_results(results_with_sw)

# --- TEST MÔ HÌNH KHÔNG STOPWORDS ---
query_clean = "kinh tế việt nam" # (Xóa các từ như "ở", "tại", "là"... nếu có)

results_no_sw = search_bm25(
    query_text=query_clean,
    k=k_results,
    bm25_model=bm25_model_no_sw,
    df_documents=df_no_sw,
    model_name="KHÔNG Stopwords"
)
display_bm25_results(results_no_sw)


**Truy vấn [CÓ Stopwords]:** 'kinh tế việt nam'
--------------------------------------------------------------------------------
Rank 1 | BM25 Score: 18.0658
Tiêu đề: Lý giải về nghi thức 'Thông dạ tế' trong lễ tang ông Abe Shinzo
Nguồn: thanhnien.vn | Chủ đề: Văn hóa
Nội dung: Lễ tang cựu Thủ tướng Nhật Bản Abe Shinzo đã được tiến hành vào lúc 18 giờ hôm nay 11.7 (16 giờ theo giờ Hà Nội) tại chùa Zojo-ji (Tăng Thượng tự) ở Tokyo, trung tâm thờ tự chính của Phật giáo Tịnh Độ...
--------------------------------------------------------------------------------
Rank 2 | BM25 Score: 16.8483
Tiêu đề: Các vua triều Nguyễn tưởng niệm ngày kinh thành Huế thất thủ thế nào?
Nguồn: danviet | Chủ đề: Đông Tây - Kim Cổ
Nội dung:  Ngày "giỗ chung" bắt đầu từ triều vua Thành Thái. Sử liệu Châu bản triều Nguyễn cho biết, năm Thành Thái thứ 6 (1894), Bộ Lễ tâu, xin xây đàn ở trong thành, hàng năm đến tế một lần. Sau đó, vua Thành...
----------------------------------------------------------------------

In [9]:
import gc

# Xóa các biến chứa văn bản thô (nếu bạn không cần chạy tìm kiếm trong session này nữa)
# Mục đích là lấy lại chỗ trống cho RAM để lưu mô hình
try:
    del tokenized_corpus_with_sw
    del tokenized_corpus_no_sw
    del df_with_sw
    del df_no_sw
except NameError:
    pass # Nếu biến đã bị xóa rồi thì bỏ qua

gc.collect()

762

In [10]:
path_model_with_sw = os.path.join(save_dir, "bm25_model_with_sw.pkl")
with open(path_model_with_sw, "wb") as f:
    pickle.dump(bm25_model_with_sw, f)

path_model_no_sw = os.path.join(save_dir, "bm25_model_no_sw.pkl")
with open(path_model_no_sw, "wb") as f:
    pickle.dump(bm25_model_no_sw, f)

print(f"Đã lưu trực tiếp 2 mô hình BM25 vào thư mục: {save_dir}")

Đã lưu trực tiếp 2 mô hình BM25 vào thư mục: models/bm25
